In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/oof_lgb_fe_v2.npy
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/oof_lgb_baseline.npy
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/submission_phase4_lgb_fe_v2.csv
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/submission_phase3_lgb_baseline.csv
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/oof_logreg.npy
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/test_fe.parquet
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__results__.html
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/features_v2.json
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/train_fe.parquet
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__notebook__.ipynb
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__output__.json
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/custom.css
/kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/__results___files/__results___15_1.png

# Phase 5 — Model Diversity

## CELL 1 — Imports, config, and Phase 4 artifact loading

In [2]:
# === CELL 1 — setup ===
from __future__ import annotations
import os, sys, gc, json, time, random, logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
import catboost as cb


# Bulletproof helper to find the exact folder containing the competition data
def get_kaggle_data_dir() -> Path:
    base_input = Path("/kaggle/input")
    if base_input.exists():
        for file_path in base_input.rglob("train.csv"):
            return file_path.parent
    return Path(".")


@dataclass(frozen=True)
class Config:
    seed: int = 42
    n_splits: int = 5
    target: str = "PitNextLap"
    id_col: str = "id"
    data_dir: Path = field(default_factory=get_kaggle_data_dir)
    out_dir: Path = field(default_factory=lambda: Path("/kaggle/working"))


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)


def get_logger(name="f1"):
    lg = logging.getLogger(name)
    if not lg.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S"))
        lg.addHandler(h); lg.setLevel(logging.INFO); lg.propagate = False
    return lg


log = get_logger()
random.seed(CFG.seed); np.random.seed(CFG.seed)


# --- Find Phase 4 artifacts ---
def find_artifact(filename: str) -> Path | None:
    # Use rglob to recursively search ALL subfolders
    for p in Path("/kaggle/input").rglob(filename):
        if p.is_file():
            return p
            
    # Fallback: working dir
    p = CFG.out_dir / filename
    return p if p.exists() else None


train_path  = find_artifact("train_fe.parquet")
test_path   = find_artifact("test_fe.parquet")
feats_path  = find_artifact("features_v2.json")

assert train_path and test_path and feats_path, \
    "Phase 4 artifacts not found. Attach the Phase 4 notebook as a data source."

log.info(f"train_fe: {train_path}")
log.info(f"test_fe:  {test_path}")
log.info(f"features_v2: {feats_path}")

train_fe = pd.read_parquet(train_path)
test_fe  = pd.read_parquet(test_path)
with open(feats_path) as f:
    feats_manifest = json.load(f)

FEATURES = feats_manifest["features"]
CAT_FEATS = feats_manifest["cat_features"]
log.info(f"Train: {train_fe.shape} | Test: {test_fe.shape} | n_features: {len(FEATURES)}")

09:52:06 | INFO | train_fe: /kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/train_fe.parquet
09:52:06 | INFO | test_fe:  /kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/test_fe.parquet
09:52:06 | INFO | features_v2: /kaggle/input/notebooks/amanthaviraj/notebook97cb90dfee/features_v2.json
09:52:07 | INFO | Train: (439140, 61) | Test: (188165, 60) | n_features: 59


## CELL 2 — Reproduce the same CV splits

In [3]:
# === CELL 2 — splits ===
# CRITICAL: use the same splitter seed/scheme as Phase 4 so OOF arrays align.
skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
SPLITS = list(skf.split(train_fe, train_fe[CFG.target]))

y = train_fe[CFG.target].astype(int).values
X = train_fe[FEATURES]
X_test = test_fe[FEATURES]
log.info(f"Splits: {CFG.n_splits} folds, total rows: {len(X)}")

09:52:07 | INFO | Splits: 5 folds, total rows: 439140


## CELL 3 — Model #1: LightGBM, tuned

In [4]:
# === CELL 3 — LightGBM tuned ===
# Mild manual tuning. For full Optuna sweep see Cell 7 (optional).

LGB_PARAMS = dict(
    n_estimators=4000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.75,
    reg_alpha=0.5,
    reg_lambda=1.0,
    min_split_gain=0.0,
    random_state=CFG.seed,
    n_jobs=-1,
    verbose=-1,
    objective="binary",
    metric="auc",
)


def train_lgb_oof(X, y, X_test, splits, params, cat_features):
    oof = np.zeros(len(X), dtype=np.float32)
    test_preds = np.zeros(len(X_test), dtype=np.float32)
    fold_scores = []
    t0 = time.time()
    for fold, (tr, va) in enumerate(splits):
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr], y[tr],
              eval_set=[(X.iloc[va], y[va])],
              categorical_feature=cat_features,
              callbacks=[lgb.early_stopping(150, verbose=False),
                         lgb.log_evaluation(0)])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        test_preds += m.predict_proba(X_test)[:, 1] / len(splits)
        s = roc_auc_score(y[va], oof[va])
        fold_scores.append(s)
        log.info(f"  [lgb] fold {fold}: AUC={s:.5f} | best_iter={m.best_iteration_} | {time.time()-t0:.0f}s")
    log.info(f"  [lgb] OOF AUC = {roc_auc_score(y, oof):.5f}")
    return oof, test_preds, fold_scores


oof_lgb, test_lgb, scores_lgb = train_lgb_oof(X, y, X_test, SPLITS, LGB_PARAMS, CAT_FEATS)

np.save(CFG.out_dir / "p5_oof_lgb.npy", oof_lgb)
np.save(CFG.out_dir / "p5_test_lgb.npy", test_lgb)

09:53:08 | INFO |   [lgb] fold 0: AUC=0.94611 | best_iter=509 | 61s
09:54:09 | INFO |   [lgb] fold 1: AUC=0.94440 | best_iter=510 | 122s
09:55:07 | INFO |   [lgb] fold 2: AUC=0.94522 | best_iter=481 | 179s
09:56:08 | INFO |   [lgb] fold 3: AUC=0.94409 | best_iter=519 | 240s
09:57:08 | INFO |   [lgb] fold 4: AUC=0.94563 | best_iter=518 | 300s
09:57:08 | INFO |   [lgb] OOF AUC = 0.94509


## CELL 4 — Model #2: XGBoost

In [5]:
# === CELL 4 — XGBoost ===
# XGBoost 2.0+ supports native categorical handling.
# Convert categoricals to pandas categorical dtype.

X_xgb = X.copy()
X_test_xgb = X_test.copy()
for c in CAT_FEATS:
    if c in X_xgb.columns:
        X_xgb[c] = X_xgb[c].astype("category")
        X_test_xgb[c] = X_test_xgb[c].astype("category")
        # Ensure same categories in train and test
        cats = X_xgb[c].cat.categories
        X_test_xgb[c] = X_test_xgb[c].cat.set_categories(cats)

XGB_PARAMS = dict(
    n_estimators=4000,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_alpha=0.5,
    reg_lambda=1.0,
    gamma=0.0,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    random_state=CFG.seed,
    n_jobs=-1,
)


def train_xgb_oof(X, y, X_test, splits, params):
    oof = np.zeros(len(X), dtype=np.float32)
    test_preds = np.zeros(len(X_test), dtype=np.float32)
    fold_scores = []
    t0 = time.time()
    for fold, (tr, va) in enumerate(splits):
        m = xgb.XGBClassifier(**params, early_stopping_rounds=150)
        m.fit(X.iloc[tr], y[tr],
              eval_set=[(X.iloc[va], y[va])],
              verbose=False)
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        test_preds += m.predict_proba(X_test)[:, 1] / len(splits)
        s = roc_auc_score(y[va], oof[va])
        fold_scores.append(s)
        log.info(f"  [xgb] fold {fold}: AUC={s:.5f} | best_iter={m.best_iteration} | {time.time()-t0:.0f}s")
    log.info(f"  [xgb] OOF AUC = {roc_auc_score(y, oof):.5f}")
    return oof, test_preds, fold_scores


oof_xgb, test_xgb, scores_xgb = train_xgb_oof(X_xgb, y, X_test_xgb, SPLITS, XGB_PARAMS)

np.save(CFG.out_dir / "p5_oof_xgb.npy", oof_xgb)
np.save(CFG.out_dir / "p5_test_xgb.npy", test_xgb)

del X_xgb, X_test_xgb; gc.collect()

09:59:04 | INFO |   [xgb] fold 0: AUC=0.94697 | best_iter=842 | 116s
10:00:52 | INFO |   [xgb] fold 1: AUC=0.94502 | best_iter=784 | 225s
10:02:20 | INFO |   [xgb] fold 2: AUC=0.94552 | best_iter=595 | 312s
10:04:22 | INFO |   [xgb] fold 3: AUC=0.94414 | best_iter=893 | 434s
10:06:17 | INFO |   [xgb] fold 4: AUC=0.94617 | best_iter=843 | 549s
10:06:17 | INFO |   [xgb] OOF AUC = 0.94555


74

## CELL 5 — Model #3: CatBoost

In [6]:
# === CELL 5 — CatBoost (GPU Optimized) ===
# CatBoost shines with high-cardinality categoricals (Driver, Race) because
# its target encoding is built-in and ordered (no leakage).

import time
import gc
import numpy as np
import catboost as cb
from sklearn.metrics import roc_auc_score

# CatBoost needs categorical columns as object/string OR int — not category dtype with categories from another df
# Re-build a CatBoost-ready frame
X_cb = X.copy()
X_test_cb = X_test.copy()
cat_idx = []
for c in CAT_FEATS:
    if c in X_cb.columns:
        # Convert to string to keep CatBoost happy
        X_cb[c] = X_cb[c].astype(str)
        X_test_cb[c] = X_test_cb[c].astype(str)
        cat_idx.append(X_cb.columns.get_loc(c))

CB_PARAMS = dict(
    iterations=4000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    bagging_temperature=0.8,
    random_strength=1.0,
    border_count=128,
    od_type="Iter",
    od_wait=150,
    eval_metric="AUC",
    loss_function="Logloss",
    random_seed=CFG.seed,
    allow_writing_files=False,
    # === NEW: GPU and Logging parameters ===
    task_type="GPU",   
    verbose=100,       
)

def train_cb_oof(X, y, X_test, splits, params, cat_idx):
    oof = np.zeros(len(X), dtype=np.float32)
    test_preds = np.zeros(len(X_test), dtype=np.float32)
    fold_scores = []
    t0 = time.time()
    
    for fold, (tr, va) in enumerate(splits):
        m = cb.CatBoostClassifier(**params)
        
        # FIX: Reverted y back to standard numpy indexing: y[tr] and y[va]
        m.fit(X.iloc[tr], y[tr],
              eval_set=(X.iloc[va], y[va]),
              cat_features=cat_idx,
              use_best_model=True)
              
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        test_preds += m.predict_proba(X_test)[:, 1] / len(splits)
        
        s = roc_auc_score(y[va], oof[va])
        fold_scores.append(s)
        log.info(f"  [cb]  fold {fold}: AUC={s:.5f} | best_iter={m.tree_count_} | {time.time()-t0:.0f}s elapsed")
        
    log.info(f"  [cb]  OOF AUC = {roc_auc_score(y, oof):.5f}")
    return oof, test_preds, fold_scores


oof_cb, test_cb, scores_cb = train_cb_oof(X_cb, y, X_test_cb, SPLITS, CB_PARAMS, cat_idx)

np.save(CFG.out_dir / "p5_oof_cb.npy", oof_cb)
np.save(CFG.out_dir / "p5_test_cb.npy", test_cb)

del X_cb, X_test_cb; gc.collect()

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9124440	best: 0.9124440 (0)	total: 199ms	remaining: 13m 17s
100:	test: 0.9352017	best: 0.9352017 (100)	total: 5.73s	remaining: 3m 41s
200:	test: 0.9393287	best: 0.9393287 (200)	total: 11s	remaining: 3m 28s
300:	test: 0.9419649	best: 0.9419649 (300)	total: 16.4s	remaining: 3m 21s
400:	test: 0.9436472	best: 0.9436472 (400)	total: 21.9s	remaining: 3m 16s
500:	test: 0.9449167	best: 0.9449167 (500)	total: 27.1s	remaining: 3m 8s
600:	test: 0.9457535	best: 0.9457535 (600)	total: 32.4s	remaining: 3m 2s
700:	test: 0.9464366	best: 0.9464366 (700)	total: 37.6s	remaining: 2m 56s
800:	test: 0.9469393	best: 0.9469393 (800)	total: 42.8s	remaining: 2m 51s
900:	test: 0.9473813	best: 0.9473813 (900)	total: 48.1s	remaining: 2m 45s
1000:	test: 0.9477441	best: 0.9477441 (1000)	total: 53.5s	remaining: 2m 40s
1100:	test: 0.9480195	best: 0.9480195 (1100)	total: 58.8s	remaining: 2m 34s
1200:	test: 0.9482657	best: 0.9482657 (1200)	total: 1m 4s	remaining: 2m 29s
1300:	test: 0.9485233	best: 0.9485233 (

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9036663	best: 0.9036663 (0)	total: 80.2ms	remaining: 5m 20s
100:	test: 0.9329411	best: 0.9329411 (100)	total: 5.57s	remaining: 3m 35s
200:	test: 0.9370071	best: 0.9370071 (200)	total: 11s	remaining: 3m 27s
300:	test: 0.9395707	best: 0.9395707 (300)	total: 16.3s	remaining: 3m 20s
400:	test: 0.9412262	best: 0.9412262 (400)	total: 21.8s	remaining: 3m 15s
500:	test: 0.9425440	best: 0.9425440 (500)	total: 27.2s	remaining: 3m 9s
600:	test: 0.9433955	best: 0.9433955 (600)	total: 32.5s	remaining: 3m 3s
700:	test: 0.9440258	best: 0.9440258 (700)	total: 37.8s	remaining: 2m 57s
800:	test: 0.9445349	best: 0.9445349 (800)	total: 43.1s	remaining: 2m 52s
900:	test: 0.9449390	best: 0.9449390 (900)	total: 48.5s	remaining: 2m 46s
1000:	test: 0.9453054	best: 0.9453054 (1000)	total: 53.9s	remaining: 2m 41s
1100:	test: 0.9456134	best: 0.9456134 (1100)	total: 59.1s	remaining: 2m 35s
1200:	test: 0.9459174	best: 0.9459174 (1200)	total: 1m 4s	remaining: 2m 30s
1300:	test: 0.9461600	best: 0.9461600 (

Default metric period is 5 because AUC is/are not implemented for GPU


...

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9042145	best: 0.9042145 (0)	total: 82.2ms	remaining: 5m 28s
100:	test: 0.9332576	best: 0.9332576 (100)	total: 5.69s	remaining: 3m 39s
200:	test: 0.9372140	best: 0.9372140 (200)	total: 11.1s	remaining: 3m 30s
300:	test: 0.9396466	best: 0.9396466 (300)	total: 16.6s	remaining: 3m 24s
400:	test: 0.9412183	best: 0.9412183 (400)	total: 22.1s	remaining: 3m 18s
500:	test: 0.9424242	best: 0.9424242 (500)	total: 27.4s	remaining: 3m 11s
600:	test: 0.9433291	best: 0.9433291 (600)	total: 32.8s	remaining: 3m 5s
700:	test: 0.9440462	best: 0.9440462 (700)	total: 38.2s	remaining: 2m 59s
800:	test: 0.9445603	best: 0.9445603 (800)	total: 43.6s	remaining: 2m 54s
900:	test: 0.9449472	best: 0.9449472 (900)	total: 49s	remaining: 2m 48s
1000:	test: 0.9453432	best: 0.9453432 (1000)	total: 54.4s	remaining: 2m 42s
1100:	test: 0.9456457	best: 0.9456457 (1100)	total: 59.7s	remaining: 2m 37s
1200:	test: 0.9458801	best: 0.9458801 (1200)	total: 1m 4s	remaining: 2m 31s
1300:	test: 0.9461133	best: 0.9461133 

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9042155	best: 0.9042155 (0)	total: 80.2ms	remaining: 5m 20s
100:	test: 0.9339462	best: 0.9339462 (100)	total: 5.56s	remaining: 3m 34s
200:	test: 0.9379671	best: 0.9379671 (200)	total: 11s	remaining: 3m 28s
300:	test: 0.9405543	best: 0.9405543 (300)	total: 16.6s	remaining: 3m 23s
400:	test: 0.9422481	best: 0.9422481 (400)	total: 22s	remaining: 3m 17s
500:	test: 0.9434600	best: 0.9434600 (500)	total: 27.3s	remaining: 3m 10s
600:	test: 0.9443160	best: 0.9443160 (600)	total: 32.6s	remaining: 3m 4s
700:	test: 0.9449674	best: 0.9449674 (700)	total: 37.9s	remaining: 2m 58s
800:	test: 0.9455085	best: 0.9455085 (800)	total: 43.2s	remaining: 2m 52s
900:	test: 0.9459691	best: 0.9459691 (900)	total: 48.5s	remaining: 2m 46s
1000:	test: 0.9463673	best: 0.9463673 (1000)	total: 53.7s	remaining: 2m 40s
1100:	test: 0.9466646	best: 0.9466646 (1100)	total: 59.1s	remaining: 2m 35s
1200:	test: 0.9469703	best: 0.9469703 (1200)	total: 1m 4s	remaining: 2m 29s
1300:	test: 0.9471980	best: 0.9471980 (1

0

## CELL 8 — Per-model submission CSVs (for sanity-checking LB ↔ OOF)

In [7]:
# === CELL 8 — per-model submissions ===
def save_sub(test_probs, name):
    out = pd.DataFrame({CFG.id_col: test_fe[CFG.id_col], CFG.target: test_probs})
    path = CFG.out_dir / f"submission_phase5_{name}.csv"
    out.to_csv(path, index=False)
    log.info(f"  saved {path.name}")

save_sub(test_lgb, "lgb")
save_sub(test_xgb, "xgb")
save_sub(test_cb,  "cb")

10:25:45 | INFO |   saved submission_phase5_lgb.csv
10:25:46 | INFO |   saved submission_phase5_xgb.csv
10:25:46 | INFO |   saved submission_phase5_cb.csv
